Assignment 4:

Load the data from multiple weather stations into Spark delta lake tables.

Dataset: Weather data files (WS01.txt, WS02.txt …. WS10.csv)

Steps Involved:

Step1: Create a folder and copy the weather history files from above link to a folder in databricks file system.

Step2: Read the weather data files one at a time using streaming data read, ensuring one file is read per trigger. (Hint: use readStream with maxFilesPerTrigger = 1)

Step3: Perform the following transformations

a. Drop the following columns : record_min_temp_year, record_max_temp_year

b. Apply Filter to remove records with zero actual_precipitation (Filter condition: actual_precipitation > 0)

c. Create a new column deviation_from_avg = abs(actual_precipitation – avg_precipitation)

Step4: Write the data to a delta table called WeatherData which is partitioned by year and month

Step 1: Create a folder and copy weather history files to DBFS

Access Databricks Workspace: Log in to your Databricks workspace.

Create a Folder: You can create a folder within the Databricks File System (DBFS) using either the Databricks UI or the dbutils.fs command in a notebook.

UI: Navigate to Data > DBFS > FileStore and create a new folder (e.g., weather_data).

dbutils.fs.mkdirs("/FileStore/weather_data")


Copy Weather Files: You'll need to manually copy the weather data files (WS01.txt, WS02.txt, etc.) from your local machine (or other location) to the newly created DBFS folder. You can achieve this using the Databricks CLI or through the UI, following steps similar to those mentioned in the search results.

Databricks CLI (example): databricks fs cp <local-file-path> dbfs:/FileStore/weather_data/<filename>. 

In [0]:
dbutils.fs.mkdirs("/FileStore/weather_data")

In [0]:
databricks fs cp <local-file-path> dbfs:/FileStore/weather_data/<filename>

Step2: Read the weather data files one at a time using streaming data read, ensuring one file is read per trigger. (Hint: use readStream with maxFilesPerTrigger = 1)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Define the schema for your weather data
weather_schema = StructType([
    StructField("station_id", StringType(), True),
    StructField("date", DateType(), True),
    StructField("actual_precipitation", DoubleType(), True),
    StructField("avg_precipitation", DoubleType(), True),
    StructField("record_min_temp_year", IntegerType(), True),
    StructField("record_max_temp_year", IntegerType(), True),
    # Add other columns from your weather data files as needed
])

# Read data from the folder using streaming
weather_stream = spark.readStream \
    .format("csv") \
    .schema(weather_schema) \
    .option("header", "true") \
    .option("maxFilesPerTrigger", 1) \
    .load("/FileStore/weather_data")

Step 3: Checking the history after the restore command and its effect on table history

After restoring the table, its data will reflect the state before the merge transaction. 
The transaction log will also reflect this restoration operation. 
New entries related to the restore will be added to the log, but previous entries prior to the restore, including the merge transaction, might not be readily available for direct querying using fn_dblog if the log has been truncated or overwritten. 

To ensure a complete and auditable history of changes (including restorations), implementing a robust auditing solution with a dedicated audit table or using SQL Server audit features is recommended.